# Table 2 핵심 변수 기술통계 재현 및 검산 노트북

이 노트북은 2024년 기업정보화통계조사 최종 분석용 데이터셋을 기준으로 중간보고서의 `Table 2A`, `Table 2B`, `Table 2C`를 재현하고 검산하기 위한 파일입니다.

원칙:
- 기술통계 산출에는 반드시 `working/analysis` 폴더 안의 최종 분석용 데이터셋만 사용합니다.
- 원자료, subset, rename, cleaned, featured 파일은 기술통계 산출에 사용하지 않습니다.
- 데이터는 변경하지 않고 읽기만 합니다.
- 변수별 유효 N, 결측 N, 결측률을 확인합니다.
- 라벨은 코드북에서 확인 가능한 경우에만 붙입니다.

## 1. 라이브러리와 경로 설정

프로젝트 루트에서 실행하는 것을 기준으로 작성했습니다. 다른 위치에서 노트북을 열어도 현재 노트북 위치를 기준으로 프로젝트 루트를 추정합니다.

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

# 노트북이 output/tables 안에 있으므로, 부모의 부모가 프로젝트 루트입니다.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / 'working').exists():
    BASE_DIR = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent.parent / 'working').exists():
    BASE_DIR = NOTEBOOK_DIR.parent.parent
else:
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다. working 폴더가 있는 위치에서 실행해 주세요.')

RAW_DIR = BASE_DIR / 'raw'
WORKING_DIR = BASE_DIR / 'working'
CODE_DIR = BASE_DIR / 'code'
ANALYSIS_DIR = WORKING_DIR / 'analysis'
OUT_DIR = BASE_DIR / 'output' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR =', BASE_DIR)
print('ANALYSIS_DIR =', ANALYSIS_DIR)
print('OUT_DIR =', OUT_DIR)

BASE_DIR = /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터
ANALYSIS_DIR = /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터/working/analysis
OUT_DIR = /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터/output/tables


## 2. 프로젝트 폴더 구조 확인

요청 조건에 맞게 `raw`, `working`, `code` 폴더와 `working` 안의 처리 단계별 폴더가 존재하는지 확인합니다.

In [20]:
required_dirs = [
    RAW_DIR,
    WORKING_DIR,
    CODE_DIR,
    WORKING_DIR / 'subset',
    WORKING_DIR / 'rename',
    WORKING_DIR / 'cleaned',
    WORKING_DIR / 'featured',
    WORKING_DIR / 'analysis',
]

folder_check = pd.DataFrame({
    'folder': [str(p.relative_to(BASE_DIR)) for p in required_dirs],
    'exists': [p.exists() and p.is_dir() for p in required_dirs],
})
folder_check

,folder,exists
0,raw,True
1,working,True
2,code,True
3,working/subset,True
4,working/rename,True
5,working/cleaned,True
6,working/featured,True
7,working/analysis,True


## 3. 전처리 스크립트 확인

`code` 폴더 안의 전처리 스크립트를 파일명 순서대로 확인합니다.

In [21]:
script_files = sorted(CODE_DIR.glob('*.py'))
scripts = pd.DataFrame({
    'script': [p.name for p in script_files],
    'modified_time': [pd.Timestamp(p.stat().st_mtime, unit='s') for p in script_files],
    'size_bytes': [p.stat().st_size for p in script_files],
})
scripts

,script,modified_time,size_bytes
0,01_make_2024_subset.py,2026-04-26 17:02:44.459339142,9464
1,02_rename_2024_variables.py,2026-04-27 10:59:30.908144474,15526
2,03_clean_2024_variables.py,2026-04-27 17:16:57.481920242,29337
3,04_make_2024_featured.py,2026-04-27 17:43:59.429408550,22976
4,05_make_2024_analysis_total.py,2026-04-27 19:01:14.266332150,13169


## 4. 최종 분석용 데이터셋 선택

최종 분석용 데이터셋은 반드시 `working/analysis` 폴더에서 찾습니다. 우선순위는 `csv`, `xlsx`, `parquet` 순서입니다. 여러 파일이 있으면 후보 목록과 수정일을 확인합니다.

In [22]:
patterns = ['*.csv', '*.xlsx', '*.parquet']
candidates = []
for priority, pattern in enumerate(patterns, start=1):
    for p in ANALYSIS_DIR.glob(pattern):
        candidates.append({
            'priority': priority,
            'file': p.name,
            'path': p,
            'modified_time': pd.Timestamp(p.stat().st_mtime, unit='s'),
            'size_bytes': p.stat().st_size,
        })

candidate_df = pd.DataFrame(candidates).sort_values(['priority', 'modified_time'], ascending=[True, False])
display(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']])

if candidate_df.empty:
    raise FileNotFoundError('working/analysis 안에서 csv/xlsx/parquet 분석 파일을 찾지 못했습니다.')

DATA_PATH = candidate_df.iloc[0]['path']
print('사용할 최종 분석용 데이터셋:', DATA_PATH.relative_to(BASE_DIR))

,priority,file,modified_time,size_bytes
0,1,nia_2024_analysis_total.csv,2026-04-27 19:01:15.774730444,1654412


사용할 최종 분석용 데이터셋: working/analysis/nia_2024_analysis_total.csv


## 5. 데이터 읽기

아래 셀은 최종 분석용 데이터셋만 읽습니다. 이후 모든 기술통계는 이 `df`에서만 계산합니다.

In [23]:
suffix = DATA_PATH.suffix.lower()
if suffix == '.csv':
    df = pd.read_csv(DATA_PATH)
elif suffix in ['.xlsx', '.xls']:
    df = pd.read_excel(DATA_PATH)
elif suffix == '.parquet':
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')

total_n = len(df)
print('사용 데이터 파일:', DATA_PATH.relative_to(BASE_DIR))
print('전체 N:', f'{total_n:,}')
print('열 수:', df.shape[1])

사용 데이터 파일: working/analysis/nia_2024_analysis_total.csv
전체 N: 12,203
열 수: 58


## 6. 변수 존재 여부와 결측 확인

분석 대상 8개 변수가 최종 분석용 데이터셋에 실제로 존재하는지 확인합니다. 존재하지 않는 변수는 새로 만들지 않고 확인 필요로 표시합니다.

In [24]:
core_var_info = [
    ('effect_proc_improve', 'Main DV', '프로세스 개선 효과', '순서형'),
    ('effect_average', 'Robustness DV', '정보화 효과 평균', '연속형 평균'),
    ('it_org_any', 'Main IV', '정보화 담당 체계 보유 여부', '이항'),
    ('ai_use', 'AI 활용 여부', 'AI 기술 및 서비스 이용 여부', '이항'),
    ('ai_use_sum', 'H2의 DV / H3의 조절변수', 'AI 활용 유형 개수', '카운트'),
    ('it_invest_sum', '보조/강건성 변수', '정보화 투자 항목 개수', '카운트'),
    ('firm_size', '통제변수', '기업 규모', '순서형/범주형'),
    ('industry', '통제변수', '업종', '명목형'),
]

exist_rows = []
for var, role, meaning, vtype in core_var_info:
    exists = var in df.columns
    if exists:
        valid_n = int(df[var].notna().sum())
        missing_n = int(df[var].isna().sum())
    else:
        valid_n = np.nan
        missing_n = np.nan
    exist_rows.append({
        '변수명': var,
        '역할': role,
        '의미': meaning,
        '변수 유형': vtype,
        '존재 여부': exists,
        '유효 N': valid_n,
        '결측 N': missing_n,
    })

variable_check = pd.DataFrame(exist_rows)
variable_check

,변수명,역할,의미,변수 유형,존재 여부,유효 N,결측 N
0,effect_proc_improve,Main DV,프로세스 개선 효과,순서형,True,12203,0
1,effect_average,Robustness DV,정보화 효과 평균,연속형 평균,True,12203,0
2,it_org_any,Main IV,정보화 담당 체계 보유 여부,이항,True,12203,0
3,ai_use,AI 활용 여부,AI 기술 및 서비스 이용 여부,이항,True,12203,0
4,ai_use_sum,H2의 DV / H3의 조절변수,AI 활용 유형 개수,카운트,True,12203,0
5,it_invest_sum,보조/강건성 변수,정보화 투자 항목 개수,카운트,True,12203,0
6,firm_size,통제변수,기업 규모,순서형/범주형,True,12203,0
7,industry,통제변수,업종,명목형,True,12203,0


## 7. 코드북 라벨 확인

`firm_size`는 원 코드북의 `D_SIZE`, `industry`는 `D_IND` 라벨을 사용합니다. 라벨을 찾지 못한 값은 추측하지 않고 `라벨 확인 필요`로 표시합니다.

In [25]:
CODEBOOK_PATH = RAW_DIR / 'nia_2024_codebook.csv'

def labels_from_codebook(path: Path, raw_var: str) -> dict:
    cb = pd.read_csv(path, encoding='utf-8-sig')
    var_col, val_col, lab_col = cb.columns[0], cb.columns[2], cb.columns[3]
    starts = cb.index[cb[var_col].astype(str).str.strip().eq(raw_var)].tolist()
    if not starts:
        return {}
    start = starts[0]
    labels = {}
    for i in range(start, len(cb)):
        first = cb.loc[i, var_col]
        if i > start and pd.notna(first) and str(first).strip():
            break
        value = cb.loc[i, val_col]
        label = cb.loc[i, lab_col]
        if pd.notna(value) and pd.notna(label):
            key = int(value) if float(value).is_integer() else float(value)
            labels[key] = str(label).strip()
    return labels

firm_size_labels = labels_from_codebook(CODEBOOK_PATH, 'D_SIZE')
industry_labels = labels_from_codebook(CODEBOOK_PATH, 'D_IND')

print('firm_size labels:', firm_size_labels)
print('industry labels:', industry_labels)

firm_size labels: {1: '10-49명', 2: '50-249명', 3: '250~999명', 4: '1000명 이상'}
industry labels: {1: '농림수산업(광업포함)', 2: '제조업', 3: '전기 등 공기조절 공급업/수도 등 원료 재생업', 4: '건설업', 5: '도매 및 소매업', 6: '운수 및 창고업', 7: '숙박 및 음식점업', 8: '정보통신업', 9: '금융 및 보험업', 10: '부동산업', 11: '전문, 과학 및 기술서비스업', 12: '사업시설관리, 사업지원 및 서비스업', 13: '교육 서비스업', 14: '보건업 및 사회복지서비스업', 15: '예술, 스포츠 및 여가관련 서비스업', 16: '수리 및 기타 개인 서비스업'}


## 8. Table 2A 계산

연속형, 순서형, 카운트형 변수는 평균, 표준편차, 최솟값, 최댓값을 계산합니다. 이항 변수는 평균을 1의 비율로 해석합니다. `industry`는 명목형이므로 Table 2A에 넣지 않고 별도 분포표로 제시합니다.

In [26]:
table2a_vars = [
    ('effect_proc_improve', 'Main DV', '순서형', '프로세스 개선 효과'),
    ('effect_average', 'Robustness DV', '연속형 평균', '정보화 효과 평균'),
    ('it_org_any', 'Main IV', '이항', '정보화 담당 체계 보유 여부'),
    ('ai_use', 'AI 활용 여부', '이항', 'AI 기술 및 서비스 이용 여부'),
    ('ai_use_sum', 'H2의 DV / H3의 조절변수', '카운트', 'AI 활용 유형 개수'),
    ('it_invest_sum', '보조/강건성 변수', '카운트', '정보화 투자 항목 개수'),
    ('firm_size', '통제변수', '순서형/범주형', '기업 규모'),
]

rows = []
for var, role, vtype, meaning in table2a_vars:
    if var not in df.columns:
        rows.append({
            '변수명': var,
            '역할': role,
            '변수 유형': vtype,
            '유효 N': np.nan,
            '결측 N': np.nan,
            '평균 또는 비율': np.nan,
            '표준편차': np.nan,
            '최솟값': np.nan,
            '최댓값': np.nan,
            '비고': '확인 필요: 최종 분석용 데이터셋에 변수 없음',
        })
        continue

    s = pd.to_numeric(df[var], errors='coerce')
    valid_n = int(s.notna().sum())
    missing_n = int(s.isna().sum())
    missing_rate = missing_n / total_n * 100
    note = [meaning]
    if vtype == '이항':
        note.append('1의 비율')
    note.append(f'결측률 {missing_rate:.3f}%')

    rows.append({
        '변수명': var,
        '역할': role,
        '변수 유형': vtype,
        '유효 N': valid_n,
        '결측 N': missing_n,
        '평균 또는 비율': round(float(s.mean()), 3) if valid_n else np.nan,
        '표준편차': round(float(s.std(ddof=1)), 3) if valid_n > 1 else np.nan,
        '최솟값': round(float(s.min()), 3) if valid_n else np.nan,
        '최댓값': round(float(s.max()), 3) if valid_n else np.nan,
        '비고': '; '.join(note),
    })

table2a = pd.DataFrame(rows)
table2a

,변수명,역할,변수 유형,유효 N,결측 N,평균 또는 비율,표준편차,최솟값,최댓값,비고
0,effect_proc_improve,Main DV,순서형,12203,0,3.974,0.764,1.0,5.0,프로세스 개선 효과; 결측률 0.000%
1,effect_average,Robustness DV,연속형 평균,12203,0,3.632,0.612,1.0,5.0,정보화 효과 평균; 결측률 0.000%
2,it_org_any,Main IV,이항,12203,0,0.520,0.500,0.0,1.0,정보화 담당 체계 보유 여부; 1의 비율; 결측률 0.000%
3,ai_use,AI 활용 여부,이항,12203,0,0.415,0.493,0.0,1.0,AI 기술 및 서비스 이용 여부; 1의 비율; 결측률 0.000%
4,ai_use_sum,H2의 DV / H3의 조절변수,카운트,12203,0,0.877,1.265,0.0,9.0,AI 활용 유형 개수; 결측률 0.000%
5,it_invest_sum,보조/강건성 변수,카운트,12203,0,3.201,1.450,1.0,8.0,정보화 투자 항목 개수; 결측률 0.000%
6,firm_size,통제변수,순서형/범주형,12203,0,1.814,0.901,1.0,4.0,기업 규모; 결측률 0.000%


## 9. Table 2B, 2C 분포표 계산

`firm_size`와 `industry`는 범주별 N과 비율을 계산합니다.

In [27]:
def distribution_table(df: pd.DataFrame, var: str, labels: dict) -> pd.DataFrame:
    counts = df[var].value_counts(dropna=False).sort_index()
    rows = []
    for value, n in counts.items():
        if pd.isna(value):
            display_value = '결측'
            label = '결측'
        else:
            key = int(value) if isinstance(value, (int, np.integer)) or (isinstance(value, float) and float(value).is_integer()) else value
            display_value = key
            label = labels.get(key, '라벨 확인 필요')
        rows.append({
            f'{var} 값': display_value,
            '라벨': label,
            'N': int(n),
            '비율(%)': round(float(n / len(df) * 100), 3),
        })
    return pd.DataFrame(rows)

table2b = distribution_table(df, 'firm_size', firm_size_labels)
table2c = distribution_table(df, 'industry', industry_labels)

display(table2b)
display(table2c)

,firm_size 값,라벨,N,비율(%)
0,1,10-49명,5666,46.431
1,2,50-249명,3784,31.009
2,3,250~999명,2113,17.315
3,4,1000명 이상,640,5.245


,industry 값,라벨,N,비율(%)
0,1,농림수산업(광업포함),477,3.909
1,2,제조업,2769,22.691
2,3,전기 등 공기조절 공급업/수도 등 원료 재생업,224,1.836
3,4,건설업,1507,12.349
4,5,도매 및 소매업,1237,10.137
5,6,운수 및 창고업,634,5.195
6,7,숙박 및 음식점업,451,3.696
7,8,정보통신업,785,6.433
8,9,금융 및 보험업,446,3.655
9,10,부동산업,295,2.417


## 10. 추가 점검

`ai_use_sum`의 평균, 분산, 0의 비율, 분산/평균 비율을 계산해 카운트형 대안모형 검토 필요성을 확인합니다. 또한 `it_org_any`, `ai_use`의 1 비율과 주요 종속변수의 평균 및 범위를 확인합니다.

In [28]:
ai_use_sum = pd.to_numeric(df['ai_use_sum'], errors='coerce')
effect_proc_improve = pd.to_numeric(df['effect_proc_improve'], errors='coerce')
effect_average = pd.to_numeric(df['effect_average'], errors='coerce')

ai_mean = float(ai_use_sum.mean())
ai_var = float(ai_use_sum.var(ddof=1))
ai_zero_pct = float((ai_use_sum == 0).sum() / ai_use_sum.notna().sum() * 100)
ai_vmr = ai_var / ai_mean

extra_check = pd.DataFrame([
    {'항목': 'ai_use_sum 평균', '값': round(ai_mean, 3)},
    {'항목': 'ai_use_sum 표본분산', '값': round(ai_var, 3)},
    {'항목': 'ai_use_sum 0의 비율(%)', '값': round(ai_zero_pct, 3)},
    {'항목': 'ai_use_sum 분산/평균 비율', '값': round(ai_vmr, 3)},
    {'항목': 'it_org_any 1 비율(%)', '값': round(float(df['it_org_any'].mean() * 100), 3)},
    {'항목': 'ai_use 1 비율(%)', '값': round(float(df['ai_use'].mean() * 100), 3)},
    {'항목': 'effect_proc_improve 평균', '값': round(float(effect_proc_improve.mean()), 3)},
    {'항목': 'effect_proc_improve 최솟값', '값': round(float(effect_proc_improve.min()), 3)},
    {'항목': 'effect_proc_improve 최댓값', '값': round(float(effect_proc_improve.max()), 3)},
    {'항목': 'effect_average 평균', '값': round(float(effect_average.mean()), 3)},
    {'항목': 'effect_average 최솟값', '값': round(float(effect_average.min()), 3)},
    {'항목': 'effect_average 최댓값', '값': round(float(effect_average.max()), 3)},
])

extra_check

,항목,값
0,ai_use_sum 평균,0.877
1,ai_use_sum 표본분산,1.601
2,ai_use_sum 0의 비율(%),58.502
3,ai_use_sum 분산/평균 비율,1.826
4,it_org_any 1 비율(%),52.036
5,ai_use 1 비율(%),41.498
6,effect_proc_improve 평균,3.974
7,effect_proc_improve 최솟값,1.000
8,effect_proc_improve 최댓값,5.000
9,effect_average 평균,3.632


## 11. 변수별 표본 손실 확인

핵심 변수들의 유효 N이 전체 N과 다른지 확인합니다. 다르면 어떤 변수에서 표본 손실이 있는지 별도로 검토해야 합니다.

In [29]:
sample_loss = variable_check.copy()
sample_loss['전체 N'] = total_n
sample_loss['표본 손실 N'] = sample_loss['전체 N'] - sample_loss['유효 N']
sample_loss[['변수명', '유효 N', '결측 N', '표본 손실 N']]

,변수명,유효 N,결측 N,표본 손실 N
0,effect_proc_improve,12203,0,0
1,effect_average,12203,0,0
2,it_org_any,12203,0,0
3,ai_use,12203,0,0
4,ai_use_sum,12203,0,0
5,it_invest_sum,12203,0,0
6,firm_size,12203,0,0
7,industry,12203,0,0


## 12. CSV와 Markdown 저장

보고서에 붙여넣기 쉽도록 CSV 3개와 해석 메모 Markdown 파일을 저장합니다. 이 저장 작업은 원자료나 분석 데이터셋을 변경하지 않습니다.

In [30]:
table2a_path = OUT_DIR / 'table2a_descriptive_stats_core.csv'
table2b_path = OUT_DIR / 'table2b_firm_size_distribution.csv'
table2c_path = OUT_DIR / 'table2c_industry_distribution.csv'

table2a.to_csv(table2a_path, index=False, encoding='utf-8-sig')
table2b.to_csv(table2b_path, index=False, encoding='utf-8-sig')
table2c.to_csv(table2c_path, index=False, encoding='utf-8-sig')

print('저장 완료:')
print('-', table2a_path.relative_to(BASE_DIR))
print('-', table2b_path.relative_to(BASE_DIR))
print('-', table2c_path.relative_to(BASE_DIR))

In [31]:
md_path = OUT_DIR / 'descriptive_stats_interpretation.md'

need_model = '필요하다' if ai_vmr > 1.5 else '크게 필요하지 않다'
interpretation = [
    f'최종 분석 표본은 총 N = {total_n:,}개 기업으로 구성된다.',
    f'전체 표본 중 정보화 담당 체계를 보유한 기업 비중은 {df["it_org_any"].mean() * 100:.1f}%이다.',
    f'AI 활용 기업 비중은 {df["ai_use"].mean() * 100:.1f}%이며, AI 활용 유형 개수의 평균은 {ai_mean:.3f}개이다.',
    f'프로세스 개선 효과의 평균은 {effect_proc_improve.mean():.3f}점으로 나타났다.',
    f'ai_use_sum의 분산/평균 비율은 {ai_vmr:.3f}로, 카운트형 대안모형 검토가 {need_model}.',
]

def markdown_table(data: pd.DataFrame) -> str:
    def fmt(value):
        if pd.isna(value):
            return ''
        if isinstance(value, float):
            if value.is_integer():
                return str(int(value))
            return f'{value:.3f}'
        return str(value).replace('|', '\\|').replace('\n', '<br>')

    columns = list(data.columns)
    lines = ['| ' + ' | '.join(columns) + ' |']
    lines.append('| ' + ' | '.join(['---'] * len(columns)) + ' |')
    for _, row in data.iterrows():
        lines.append('| ' + ' | '.join(fmt(row[col]) for col in columns) + ' |')
    return '\n'.join(lines)

md_lines = []
md_lines.append('# 기술통계 산출 결과')
md_lines.append('')
md_lines.append(f'- 사용 파일: `{DATA_PATH.relative_to(BASE_DIR)}`')
md_lines.append(f'- 전체 N: {total_n:,}')
md_lines.append('')
md_lines.append('## Table 2A. 핵심 변수 기술통계')
md_lines.append(markdown_table(table2a))
md_lines.append('')
md_lines.append('## Table 2B. 기업 규모 분포')
md_lines.append(markdown_table(table2b))
md_lines.append('')
md_lines.append('## Table 2C. 업종 분포')
md_lines.append(markdown_table(table2c))
md_lines.append('')
md_lines.append('## ai_use_sum 과분산 점검')
md_lines.append(f'- 평균: {ai_mean:.3f}')
md_lines.append(f'- 표본분산: {ai_var:.3f}')
md_lines.append(f'- 0의 비율: {ai_zero_pct:.3f}%')
md_lines.append(f'- 분산/평균 비율: {ai_vmr:.3f}')
md_lines.append('')
md_lines.append('## 보고서용 해석 메모')
md_lines.extend(f'- {sentence}' for sentence in interpretation)

md_path.write_text('\n'.join(md_lines) + '\n', encoding='utf-8')
# print('\n'.join(md_lines) + '\n')

print('-', md_path.relative_to(BASE_DIR))


- output/tables/descriptive_stats_interpretation.md


## 13. 보고서용 해석 메모 확인

아래 문장을 중간보고서 예비분석 결과 섹션에 맞게 다듬어 사용할 수 있습니다.

In [32]:
for sentence in interpretation:
    print(sentence)

최종 분석 표본은 총 N = 12,203개 기업으로 구성된다.
전체 표본 중 정보화 담당 체계를 보유한 기업 비중은 52.0%이다.
AI 활용 기업 비중은 41.5%이며, AI 활용 유형 개수의 평균은 0.877개이다.
프로세스 개선 효과의 평균은 3.974점으로 나타났다.
ai_use_sum의 분산/평균 비율은 1.826로, 카운트형 대안모형 검토가 필요하다.
